# Model Setup & Training

## Project Context
This notebook trains and evaluates classification models on both datasets prepared in Notebook 03:
1. **Pathway A (Feature Selection):** `dataset_for_modeling_features.csv` — interpretable features for business insights.
2. **Pathway B (PCA):** `dataset_for_modeling_pca.csv` — dimensionality-reduced features for performance comparison.

**Models evaluated:**
- Logistic Regression
- Random Forest
- Gradient Boosting (XGBoost)

**Key considerations:**
- The target is imbalanced (~26.5% churn), so we use **stratified splits** and evaluate with **F1-score**, **Precision**, **Recall**, and **AUC-ROC** alongside accuracy.
- We apply **SMOTE** on the training set only to address class imbalance without leaking information into the test set.

**Note:** Detailed performance analysis, hyperparameter tuning, and remodelling will follow in subsequent notebooks.

In [ ]:
#Import fundamental libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Import Machine Learning tools
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, roc_auc_score, accuracy_score,
    roc_curve
)
from imblearn.over_sampling import SMOTE

#Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All libraries imported successfully.")

## 1. Load Datasets & Prepare Train/Test Splits

We load both pathway datasets, encode the target variable (`Churn`) as binary (Yes=1, No=0), and create stratified 80/20 train-test splits. SMOTE is then applied **only** to the training sets to balance the class distribution.

In [ ]:
#Load both datasets
df_features = pd.read_csv("dataset_for_modeling_features.csv")
df_pca = pd.read_csv("dataset_for_modeling_pca.csv")

print(f"Pathway A (Features) shape: {df_features.shape}")
print(f"Pathway B (PCA) shape:      {df_pca.shape}")

#Encode target variable: Yes=1, No=0
df_features["Churn"] = df_features["Churn"].map({"Yes": 1, "No": 0})
df_pca["Churn"] = df_pca["Churn"].map({"Yes": 1, "No": 0})

print(f"\nTarget distribution:\n{df_features['Churn'].value_counts(normalize=True).round(4)}")

In [ ]:
#Separate features and target for both pathways
X_feat = df_features.drop(columns=["Churn"])
y_feat = df_features["Churn"]

X_pca = df_pca.drop(columns=["Churn"])
y_pca = df_pca["Churn"]

#Stratified 80/20 train-test split (same random state for reproducibility)
RANDOM_STATE = 42

X_feat_train, X_feat_test, y_feat_train, y_feat_test = train_test_split(
    X_feat, y_feat, test_size=0.2, stratify=y_feat, random_state=RANDOM_STATE
)

X_pca_train, X_pca_test, y_pca_train, y_pca_test = train_test_split(
    X_pca, y_pca, test_size=0.2, stratify=y_pca, random_state=RANDOM_STATE
)

print(f"Pathway A — Train: {X_feat_train.shape[0]}, Test: {X_feat_test.shape[0]}")
print(f"Pathway B — Train: {X_pca_train.shape[0]}, Test: {X_pca_test.shape[0]}")

In [ ]:
#Apply SMOTE to training sets only (prevents data leakage into test set)
smote = SMOTE(random_state=RANDOM_STATE)

X_feat_train_sm, y_feat_train_sm = smote.fit_resample(X_feat_train, y_feat_train)
X_pca_train_sm, y_pca_train_sm = smote.fit_resample(X_pca_train, y_pca_train)

print("SMOTE applied to training sets only.\n")
print(f"Pathway A — Before SMOTE: {y_feat_train.value_counts().to_dict()}")
print(f"Pathway A — After SMOTE:  {y_feat_train_sm.value_counts().to_dict()}\n")
print(f"Pathway B — Before SMOTE: {y_pca_train.value_counts().to_dict()}")
print(f"Pathway B — After SMOTE:  {y_pca_train_sm.value_counts().to_dict()}")

## 2. Define Models & Evaluation Framework

We define the three models with sensible baseline configurations and create a reusable evaluation function that computes all key metrics. This function will be used consistently across both pathways.

In [ ]:
#Define model configurations
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        use_label_encoder=False
    )
}

def evaluate_model(model, X_test, y_test):
    """Evaluate a trained model and return a dictionary of key metrics."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "AUC-ROC": roc_auc_score(y_test, y_proba)
    }

print("Models and evaluation framework defined.")

## 3. Pathway A — Train & Evaluate on Feature-Selected Data

This pathway uses the interpretable feature set. Results here will support business insights and actionable recommendations.

In [ ]:
#Train and evaluate all models on Pathway A (Feature Selection)
results_feat = {}
trained_models_feat = {}

for name, model in models.items():
    print(f"Training {name} on Pathway A...")
    model.fit(X_feat_train_sm, y_feat_train_sm)
    trained_models_feat[name] = model
    results_feat[name] = evaluate_model(model, X_feat_test, y_feat_test)
    print(f"  ✓ {name} complete.\n")

#Display results as a DataFrame
df_results_feat = pd.DataFrame(results_feat).T.round(4)
df_results_feat.index.name = "Model"
print("=" * 60)
print("PATHWAY A — Feature Selection Results")
print("=" * 60)
df_results_feat

In [ ]:
#Classification Reports for Pathway A
for name, model in trained_models_feat.items():
    y_pred = model.predict(X_feat_test)
    print(f"\n{'=' * 50}")
    print(f"Classification Report — {name} (Pathway A)")
    print(f"{'=' * 50}")
    print(classification_report(y_feat_test, y_pred, target_names=["No Churn", "Churn"]))

In [ ]:
#Confusion Matrices for Pathway A
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confusion Matrices — Pathway A (Feature Selection)", fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, trained_models_feat.items()):
    y_pred = model.predict(X_feat_test)
    ConfusionMatrixDisplay.from_predictions(
        y_feat_test, y_pred,
        display_labels=["No Churn", "Churn"],
        cmap="Blues",
        ax=ax
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
#ROC Curves for Pathway A
plt.figure(figsize=(10, 7))

for name, model in trained_models_feat.items():
    y_proba = model.predict_proba(X_feat_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_feat_test, y_proba)
    auc = roc_auc_score(y_feat_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
plt.title("ROC Curves — Pathway A (Feature Selection)", fontsize=14)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Pathway B — Train & Evaluate on PCA Data

This pathway uses the PCA-reduced feature set (15 principal components retaining 90% variance). Results here allow us to assess whether dimensionality reduction improves or degrades model performance compared to Pathway A.

In [ ]:
#Re-initialize models for Pathway B (fresh instances to avoid state carryover)
models_pca = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        use_label_encoder=False
    )
}

#Train and evaluate all models on Pathway B (PCA)
results_pca = {}
trained_models_pca = {}

for name, model in models_pca.items():
    print(f"Training {name} on Pathway B...")
    model.fit(X_pca_train_sm, y_pca_train_sm)
    trained_models_pca[name] = model
    results_pca[name] = evaluate_model(model, X_pca_test, y_pca_test)
    print(f"  ✓ {name} complete.\n")

#Display results as a DataFrame
df_results_pca = pd.DataFrame(results_pca).T.round(4)
df_results_pca.index.name = "Model"
print("=" * 60)
print("PATHWAY B — PCA Results")
print("=" * 60)
df_results_pca

In [ ]:
#Classification Reports for Pathway B
for name, model in trained_models_pca.items():
    y_pred = model.predict(X_pca_test)
    print(f"\n{'=' * 50}")
    print(f"Classification Report — {name} (Pathway B)")
    print(f"{'=' * 50}")
    print(classification_report(y_pca_test, y_pred, target_names=["No Churn", "Churn"]))

In [ ]:
#Confusion Matrices for Pathway B
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confusion Matrices — Pathway B (PCA)", fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, trained_models_pca.items()):
    y_pred = model.predict(X_pca_test)
    ConfusionMatrixDisplay.from_predictions(
        y_pca_test, y_pred,
        display_labels=["No Churn", "Churn"],
        cmap="Oranges",
        ax=ax
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
#ROC Curves for Pathway B
plt.figure(figsize=(10, 7))

for name, model in trained_models_pca.items():
    y_proba = model.predict_proba(X_pca_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_pca_test, y_proba)
    auc = roc_auc_score(y_pca_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
plt.title("ROC Curves — Pathway B (PCA)", fontsize=14)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Pathway Comparison — Feature Selection vs. PCA

We consolidate results from both pathways into a single comparison to determine which feature representation yields better performance for each model.

In [ ]:
#Build side-by-side comparison table
comparison_rows = []
for model_name in models.keys():
    for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]:
        feat_val = results_feat[model_name][metric]
        pca_val = results_pca[model_name][metric]
        comparison_rows.append({
            "Model": model_name,
            "Metric": metric,
            "Pathway A (Features)": round(feat_val, 4),
            "Pathway B (PCA)": round(pca_val, 4),
            "Difference (A - B)": round(feat_val - pca_val, 4)
        })

df_comparison = pd.DataFrame(comparison_rows)
print("=" * 75)
print("FULL COMPARISON — Pathway A (Feature Selection) vs. Pathway B (PCA)")
print("=" * 75)
df_comparison

In [ ]:
#Visual comparison: F1-Score and AUC-ROC side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

model_names = list(models.keys())
x = np.arange(len(model_names))
bar_width = 0.35

#F1-Score comparison
f1_feat = [results_feat[m]["F1-Score"] for m in model_names]
f1_pca = [results_pca[m]["F1-Score"] for m in model_names]

axes[0].bar(x - bar_width/2, f1_feat, bar_width, label="Pathway A (Features)", color="#4C72B0")
axes[0].bar(x + bar_width/2, f1_pca, bar_width, label="Pathway B (PCA)", color="#DD8452")
axes[0].set_title("F1-Score Comparison", fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, fontsize=10)
axes[0].set_ylabel("F1-Score")
axes[0].set_ylim(0, 1)
axes[0].legend()

#Add value labels
for i, (v1, v2) in enumerate(zip(f1_feat, f1_pca)):
    axes[0].text(i - bar_width/2, v1 + 0.01, f"{v1:.3f}", ha="center", fontsize=9)
    axes[0].text(i + bar_width/2, v2 + 0.01, f"{v2:.3f}", ha="center", fontsize=9)

#AUC-ROC comparison
auc_feat = [results_feat[m]["AUC-ROC"] for m in model_names]
auc_pca = [results_pca[m]["AUC-ROC"] for m in model_names]

axes[1].bar(x - bar_width/2, auc_feat, bar_width, label="Pathway A (Features)", color="#4C72B0")
axes[1].bar(x + bar_width/2, auc_pca, bar_width, label="Pathway B (PCA)", color="#DD8452")
axes[1].set_title("AUC-ROC Comparison", fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, fontsize=10)
axes[1].set_ylabel("AUC-ROC")
axes[1].set_ylim(0, 1)
axes[1].legend()

#Add value labels
for i, (v1, v2) in enumerate(zip(auc_feat, auc_pca)):
    axes[1].text(i - bar_width/2, v1 + 0.01, f"{v1:.3f}", ha="center", fontsize=9)
    axes[1].text(i + bar_width/2, v2 + 0.01, f"{v2:.3f}", ha="center", fontsize=9)

plt.suptitle("Model Performance — Pathway A vs. Pathway B", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Feature Importance — Pathway A

Since Pathway A retains the original interpretable features, we can extract feature importance from the tree-based models and coefficients from Logistic Regression. This is essential for generating business insights about what drives customer churn.

In [ ]:
#Feature Importance: Logistic Regression (coefficients)
lr_model = trained_models_feat["Logistic Regression"]
lr_importance = pd.Series(lr_model.coef_[0], index=X_feat.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

#Top positive and negative coefficients
lr_importance.plot(kind="barh", ax=axes[0], color=lr_importance.apply(lambda x: "#C44E52" if x > 0 else "#4C72B0"))
axes[0].set_title("Logistic Regression — Feature Coefficients", fontsize=13)
axes[0].set_xlabel("Coefficient Value")
axes[0].axvline(x=0, color="black", linewidth=0.8)

#Feature Importance: Random Forest
rf_model = trained_models_feat["Random Forest"]
rf_importance = pd.Series(rf_model.feature_importances_, index=X_feat.columns).sort_values(ascending=True)

rf_importance.plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Random Forest — Feature Importance", fontsize=13)
axes[1].set_xlabel("Importance")

plt.suptitle("Feature Importance — Pathway A", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#Feature Importance: XGBoost
xgb_model = trained_models_feat["XGBoost"]
xgb_importance = pd.Series(xgb_model.feature_importances_, index=X_feat.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
xgb_importance.plot(kind="barh", color="#8172B2")
plt.title("XGBoost — Feature Importance (Pathway A)", fontsize=14)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 7. Export Results for Subsequent Notebooks

We save the comparison results and trained model objects so they can be loaded directly in the upcoming **Performance & Testing** and **Remodelling** notebooks without retraining.

In [ ]:
import joblib

#Export trained models for reuse in subsequent notebooks
joblib.dump(trained_models_feat, "trained_models_features.pkl")
joblib.dump(trained_models_pca, "trained_models_pca.pkl")

#Export test sets for consistent evaluation in subsequent notebooks
joblib.dump({
    "X_feat_test": X_feat_test, "y_feat_test": y_feat_test,
    "X_feat_train_sm": X_feat_train_sm, "y_feat_train_sm": y_feat_train_sm,
    "X_pca_test": X_pca_test, "y_pca_test": y_pca_test,
    "X_pca_train_sm": X_pca_train_sm, "y_pca_train_sm": y_pca_train_sm
}, "train_test_splits.pkl")

#Export comparison results
df_comparison.to_csv("model_comparison_results.csv", index=False)
df_results_feat.to_csv("pathway_a_results.csv")
df_results_pca.to_csv("pathway_b_results.csv")

print("Exported artifacts:")
print("  • trained_models_features.pkl  — Pathway A trained models")
print("  • trained_models_pca.pkl       — Pathway B trained models")
print("  • train_test_splits.pkl        — Train/test data splits")
print("  • model_comparison_results.csv — Full comparison table")
print("  • pathway_a_results.csv        — Pathway A metrics")
print("  • pathway_b_results.csv        — Pathway B metrics")

## Conclusion & Next Steps

This notebook established baseline models for both data pathways using Logistic Regression, Random Forest, and XGBoost. All models were trained on SMOTE-balanced training data and evaluated on the original (imbalanced) test set to reflect real-world conditions.

**Key takeaways from this notebook:**
- All three models have been trained and evaluated across both pathways with consistent metrics.
- Feature importance analysis from Pathway A provides initial insight into churn drivers (e.g., Contract type, tenure, MonthlyCharges).
- The comparison table highlights where each pathway excels or falls short.

**Artifacts exported for downstream notebooks:**
- Trained model objects (`.pkl`) — ready for deeper evaluation without retraining.
- Train/test splits (`.pkl`) — ensures consistent data across all notebooks.
- Results CSVs — for reporting and further analysis.

**Next Steps:**
1. **Notebook 05 — Performance & Testing:** Cross-validation, threshold tuning, statistical significance testing, and deeper error analysis.
2. **Notebook 06 — Remodelling:** Hyperparameter tuning (GridSearch/RandomizedSearch), ensemble strategies, and final model selection with business recommendations.